# 04 · Retrieval and reranking\n\nHybrid search = dense (meaning) + BM25 (exact words), merged with Reciprocal Rank Fusion; then the reranker scores each candidate 0-1 and a threshold decides what reaches the LLM. If nothing passes, the system refuses **before** calling the LLM.

In [1]:
import os, sys, uuid, json, asyncio
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
os.chdir(ROOT / "backend")
from app.config import Settings
from app.services.container import build_container
LIVE = bool(os.getenv("COHERE_API_KEY") and os.getenv("ANTHROPIC_API_KEY"))
settings = Settings(use_fake_providers=not LIVE, qdrant_url=None, qdrant_path=None)
c = build_container(settings)
print("MODE:", "LIVE (Cohere + Claude)" if LIVE else "FAKE providers - shows the mechanics only, not real quality")

MODE: FAKE providers - shows the mechanics only, not real quality


In [2]:
from app.ingestion.pipeline import process_file
from app.clients.sparse import doc_sparse
from app.guardrails import scan_document_text
CORPUS = ROOT / "data" / "corpus"
await c.store.ensure_collection()
DOCS = {}
for name in ["employee_handbook_2026.pdf", "travel_expense_policy.docx", "it_security_standard.pdf", "remote_work_policy_2024.docx"]:
    res = process_file(name, (CORPUS / name).read_bytes(), document_id=str(uuid.uuid4()), settings=settings)
    dense = await c.embedder.embed_documents([ch.embed_text for ch in res.chunks])
    await c.store.upsert(res.chunks, dense, [doc_sparse(ch.embed_text) for ch in res.chunks], "alice",
                         [scan_document_text(ch.text) for ch in res.chunks])
    DOCS[name] = res
    print(f"{name:32s} {len(res.chunks):>2} chunks  vector dim={len(dense[0])}")

employee_handbook_2026.pdf        9 chunks  vector dim=256
travel_expense_policy.docx        4 chunks  vector dim=256


it_security_standard.pdf          3 chunks  vector dim=256
remote_work_policy_2024.docx      2 chunks  vector dim=256


## 1. One question, every score

In [3]:
res = await c.retriever.retrieve("What is the per diem for Tier 1 cities?", "alice")
print(f"{'kept':5s} {'dense#':>6s} {'bm25#':>6s} {'rrf':>7s} {'rerank':>7s}  source")
for ch in res.candidates[:8]:
    print(f"{str(ch.kept):5s} {ch.dense_rank!s:>6s} {ch.bm25_rank!s:>6s} {ch.rrf_score:7.4f} {ch.rerank_score or 0:7.3f}  {ch.title}")
print("timings:", res.timings_ms)

kept  dense#  bm25#     rrf  rerank  source
True       1      1  0.0328   0.800  travel_expense_policy.docx, Travel and Expense Policy > Per diem
True       8      2  0.0308   0.200  employee_handbook_2026.pdf, p. 3
True       3      8  0.0306   0.200  employee_handbook_2026.pdf, p. 3
True       7      6  0.0301   0.200  employee_handbook_2026.pdf, p. 2
True      10      5  0.0297   0.200  employee_handbook_2026.pdf, p. 2
True      15      3  0.0292   0.200  remote_work_policy_2024.docx, Remote Work Policy (2024) > Allowances
False     12      7  0.0288   0.200  travel_expense_policy.docx, Travel and Expense Policy > Claims
False     16      4  0.0288   0.200  remote_work_policy_2024.docx, Remote Work Policy (2024) > Eligibility
timings: {'embed_ms': 0, 'search_ms': 3, 'rerank_ms': 0, 'retrieval_ms': 4}


## 2. Reciprocal Rank Fusion by hand\nscore = Σ 1/(60 + rank). A chunk ranked well by both lists wins.

In [4]:
from app.retrieval import reciprocal_rank_fusion
print(reciprocal_rank_fusion([["a", "b", "c"], ["c", "a", "d"]]))

{'a': 0.03252247488101534, 'b': 0.016129032258064516, 'c': 0.032266458495966696, 'd': 0.015873015873015872}


## 3. Hybrid vs dense-only on an exact-term question

In [5]:
from dataclasses import replace
from app.retrieval import Retriever
q = "Which password manager is approved, Vaultly?"
for hybrid in (True, False):
    r = Retriever(c.embedder, c.store, c.reranker, replace(c.retriever.config, hybrid=hybrid))
    out = await r.retrieve(q, "alice")
    print(f"hybrid={hybrid!s:5s} top-1: {out.candidates[0].title:40s} kept={len(out.kept)}")

hybrid=True  top-1: it_security_standard.pdf, p. 1           kept=3
hybrid=False top-1: it_security_standard.pdf, p. 1           kept=3


## 4. An unrelated question: nothing passes the threshold -> refusal without an LLM call

In [6]:
out = await c.retriever.retrieve("Who won the football world cup?", "alice")
print("kept chunks:", len(out.kept), "-> the API answers with a 'not found in your documents' refusal")

kept chunks: 0 -> the API answers with a 'not found in your documents' refusal
